# SC-DCOPF Validation with pandapower

Independent cross-validation of the custom SC-DCOPF. The network was built based on tha same data and the DC-OPF was solved with **pandapower's** `rundcopp`.

Two validations, same network:
1. **Base case** (C = {0}) → reference **$93,152.38/h**.
2. **N-1 coupled** (config `contingencies`, `redispatch_with_exclusions`) → reference
   **$93,509.32/h**, validated by decomposition.

**Why decomposition is valid:** The custom model has per-scenario variables `P[g,c]`, `f[l,c]`,
`theta[i,c]` and writes every constraint independently per scenario. No constraint couples
scenarios; they meet only in the
objective as a weighted sum. For a separable objective over independent feasible sets,
`min Σ ρ_c·cost_c = Σ ρ_c·(min cost_c)`. So the coupled optimum equals the weighted sum of each
scenario solved alone, which is what pandapower can do (one network, one dispatch at a time).

In [ ]:
import numpy as np
import pandas as pd
import pandapower as pp

# Shared processed data lives two levels up (project/processed/<source>/).
ROOT   = ".."
SOURCE = "matpower"
PROC   = f"{ROOT}/processed/{SOURCE}"

gens     = pd.read_csv(f"{PROC}/gens.csv")
loads    = pd.read_csv(f"{PROC}/loads.csv")
branches = pd.read_csv(f"{PROC}/branches.csv")
meta     = pd.read_csv(f"{PROC}/system_meta.csv").set_index("key")["value"]

BASE_MVA  = float(meta["base_mva"])
SLACK_BUS = int(meta["slack_bus"])
VN_KV     = 138.0   # single nominal voltage (cancels out in DC; fixed for consistency)
SQRT3     = np.sqrt(3.0)

print(f"base {BASE_MVA:.0f} MVA | slack {SLACK_BUS} | Vn {VN_KV} kV")
print(f"generators {len(gens)} | buses {len(loads)} | branches {len(branches)}")

## 1. Network builder

One function builds the pandapower network from the CSVs, used by both validations. Every
branch is created individually by `branch_id`, so parallel lines (same from/to) are two
independent lines — exactly as in the custom model. Returns the net plus `gen_id -> idx` and
`branch_id -> idx` lookups so we can switch elements out of service for the N-1 sweep.

In [ ]:
def build_net():
    net = pp.create_empty_network(sn_mva=BASE_MVA)

    # buses: one per MATPOWER bus number (1-indexed) -> pandapower index
    bus_idx = {}
    for b in loads["bus"].tolist():
        bus_idx[int(b)] = pp.create_bus(net, vn_kv=VN_KV, name=f"bus_{int(b)}")

    # loads: pd_pu -> MW, fixed
    for r in loads.itertuples():
        if r.pd_pu != 0.0:
            pp.create_load(net, bus=bus_idx[int(r.bus)], p_mw=r.pd_pu * BASE_MVA,
                           controllable=False, name=f"load_{int(r.bus)}")

    # generators: 19 real units; pmin/pmax pu -> MW. None is the slack as an external grid is used.
    gen_pp_idx = {}
    for r in gens.itertuples():
        gen_pp_idx[int(r.gen_id)] = pp.create_gen(
            net, bus=bus_idx[int(r.bus)], p_mw=0.0,
            min_p_mw=r.pmin_pu * BASE_MVA, max_p_mw=r.pmax_pu * BASE_MVA,
            slack=False, name=f"gen_{int(r.gen_id)}_bus_{int(r.bus)}")

    # angle reference: zero-injection ext_grid at the slack bus (fixes theta=0, injects 0 MW)
    pp.create_ext_grid(net, bus=bus_idx[SLACK_BUS], vm_pu=1.0, max_p_mw=0.0, min_p_mw=0.0)

    # branches: reactance-only impedance + MW cap calibrated so 100% loading == rate_MVA
    line_pp_idx = {}
    for r in branches.itertuples():
        rate_mva = r.rate_pu * BASE_MVA
        x_ohm    = r.x_pu * (VN_KV ** 2) / BASE_MVA
        max_i_ka = rate_mva / (SQRT3 * VN_KV)
        line_pp_idx[int(r.branch_id)] = pp.create_line_from_parameters(
            net, from_bus=bus_idx[int(r.from_bus)], to_bus=bus_idx[int(r.to_bus)],
            length_km=1.0, r_ohm_per_km=0.0, x_ohm_per_km=x_ohm, c_nf_per_km=0.0,
            max_i_ka=max_i_ka, max_loading_percent=100.0, name=f"br_{int(r.branch_id)}_{r.label}")

    # costs: linear, cp1 = c1 ($/MWh); cp0 = cp2 = 0
    for r in gens.itertuples():
        pp.create_poly_cost(net, element=gen_pp_idx[int(r.gen_id)], et="gen",
                            cp1_eur_per_mw=float(r.c1), cp0_eur=0.0)

    return net, gen_pp_idx, line_pp_idx

net0, gen_pp_idx, line_pp_idx = build_net()
print(f"builder OK: {len(net0.bus)} buses, {len(net0.gen)} gens, "
      f"{len(net0.line)} lines, {len(net0.poly_cost)} costs, {len(net0.ext_grid)} ext_grid")

## 2. Base-case validation (C = {0})

Plain DC-OPF on the full network. Should reproduce the custom base case: $93,152.38/h,
4242 MW, 13 of 19 generators dispatched, same congested lines.

In [ ]:
net, gen_pp_idx, line_pp_idx = build_net()
pp.rundcopp(net)

total_load_mw = (loads["pd_pu"] * BASE_MVA).sum()
cost_base   = float(net.res_cost)
gen_total   = net.res_gen["p_mw"].sum()
n_dispatched = int((net.res_gen["p_mw"] > 1e-3).sum())

REF_BASE = 93152.38
print(f"cost             : ${cost_base:,.2f}/h   (ref {REF_BASE:,.2f}, diff {cost_base-REF_BASE:+,.2f})")
print(f"total generation : {gen_total:,.1f} MW   (ref 4242)")
print(f"total load       : {total_load_mw:,.1f} MW")
print(f"generators > 0   : {n_dispatched} of {len(net.gen)}   (ref 13)")

In [ ]:
# dispatch per generator (merit-order check): cheap units first, expensive at zero
disp = gens.copy()
disp["pmax_mw"]   = disp["pmax_pu"] * BASE_MVA
disp["p_mw"]      = [net.res_gen.loc[gen_pp_idx[int(g)], "p_mw"] for g in disp["gen_id"]]
disp["loading_%"] = 100.0 * disp["p_mw"] / disp["pmax_mw"]
disp.sort_values("c1").reset_index(drop=True)[["gen_id", "bus", "c1", "pmax_mw", "p_mw", "loading_%"]].round(2)

In [ ]:
# congested lines (should match custom: 49-69 & 100-103 at 100%, 89-92 ~99.5%, 47-69, 94-100)
ll = branches.copy()
ll["p_from_mw"] = [net.res_line.loc[line_pp_idx[int(b)], "p_from_mw"] for b in ll["branch_id"]]
ll["loading_%"] = [net.res_line.loc[line_pp_idx[int(b)], "loading_percent"] for b in ll["branch_id"]]
ll["rate_mva"]  = ll["rate_pu"] * BASE_MVA
ll.sort_values("loading_%", ascending=False)[["branch_id", "label", "rate_mva", "p_from_mw", "loading_%"]].head(10).round(2)

## 3. N-1 validation by decomposition (config `contingencies`)

Solve one DC-OPF per operating state and recompose the expected cost `Σ ρ_c·cost_c`. The
scenario set must match the custom model exactly:
- base case (all in service)
- line contingencies: non-radial lines (`is_bridge == False`), **excluding** branch 7 (8-5)
  and branch 50 (38-37)
- generator contingencies: all 19 generators **excluding** gen 0 (bus 10)
- weights: `rho_line = 1e-3`, `rho_gen = 1e-2`, base takes the remainder

The 3 excluded elements are the ones the custom vulnerability sweep found to have no corrective
solution; removing them is the `redispatch_with_exclusions` formulation.

In [ ]:
RHO_GEN  = 1e-2
RHO_LINE = 1e-3
EXCLUDE_LINES = {7, 50}   # 8-5, 38-37
EXCLUDE_GENS  = {0}       # bus 10 (505 MW)

CT_lines = [int(l) for l in branches.loc[~branches["is_bridge"], "branch_id"].tolist()
            if int(l) not in EXCLUDE_LINES]
CG_gens  = [int(g) for g in gens["gen_id"].tolist() if int(g) not in EXCLUDE_GENS]

scenarios = [{"id": 0, "type": "base", "out_line": None, "out_gen": None, "rho": None}]
for l in CT_lines:
    scenarios.append({"id": len(scenarios), "type": "line", "out_line": l, "out_gen": None, "rho": RHO_LINE})
for g in CG_gens:
    scenarios.append({"id": len(scenarios), "type": "gen", "out_line": None, "out_gen": g, "rho": RHO_GEN})

rho_cont_sum = sum(s["rho"] for s in scenarios if s["rho"] is not None)
scenarios[0]["rho"] = 1.0 - rho_cont_sum
assert scenarios[0]["rho"] > 0 and abs(sum(s["rho"] for s in scenarios) - 1.0) < 1e-9

print(f"|C| = {len(scenarios)} states (1 base + {len(CT_lines)} lines + {len(CG_gens)} gens) — expect 194")
print(f"rho_base = {scenarios[0]['rho']:.4f} | contingencies sum = {rho_cont_sum:.4f}")

In [ ]:
# one DC-OPF per scenario (rebuild a clean net each time, switch the contingent element out)
rows = []
for s in scenarios:
    net, gidx, lidx = build_net()
    if s["out_line"] is not None:
        net.line.loc[lidx[s["out_line"]], "in_service"] = False
    if s["out_gen"] is not None:
        net.gen.loc[gidx[s["out_gen"]], "in_service"] = False
    try:
        pp.rundcopp(net)
        cost, feasible = float(net.res_cost), True
    except Exception:
        cost, feasible = np.nan, False
    rows.append({"id": s["id"], "type": s["type"], "out_line": s["out_line"],
                 "out_gen": s["out_gen"], "rho": s["rho"], "feasible": feasible, "cost": cost})

res = pd.DataFrame(rows)
n_infeasible = int((~res["feasible"]).sum())
print(f"solved {len(res)} scenarios | infeasible: {n_infeasible}")
if n_infeasible:
    print(res[~res["feasible"]][["id", "type", "out_line", "out_gen"]].to_string(index=False))

In [ ]:
# recompose the expected cost and compare with the custom coupled model
REF_EXPECTED = 93509.32
assert res["feasible"].all(), "a scenario is infeasible; decomposition sum undefined"

expected_cost = float((res["rho"] * res["cost"]).sum())
base_cost     = float(res.loc[res["id"] == 0, "cost"].iloc[0])

print(f"{'metric':<26}{'pandapower':>16}{'custom':>16}{'diff':>14}")
print("-" * 72)
print(f"{'base case ($/h)':<26}{base_cost:>16,.2f}{REF_BASE:>16,.2f}{base_cost-REF_BASE:>13,.2f}")
print(f"{'expected cost ($/h)':<26}{expected_cost:>16,.2f}{REF_EXPECTED:>16,.2f}{expected_cost-REF_EXPECTED:>13,.2f}")
print("-" * 72)
print(f"expected-cost difference: {expected_cost-REF_EXPECTED:+,.2f} $/h  "
      f"({100*(expected_cost-REF_EXPECTED)/REF_EXPECTED:+.4f}%)")

In [ ]:
# per-scenario cost by type (compare with RESULTS_NOTES.md: gen contingencies ~4x line ones)
for t in ["base", "line", "gen"]:
    sub = res[res["type"] == t]
    if len(sub):
        print(f"{t:<6} n={len(sub):>3}  mean=${sub['cost'].mean():>10,.2f}  "
              f"max=${sub['cost'].max():>10,.2f}  min=${sub['cost'].min():>10,.2f}")